# 05 · 법령 조문 스냅샷 (계층 1)

| | |
|---|---|
| **목적** | 근거 3계층의 **계층 1 = 사실**. 처분 판정이 인용하는 조문의 무결성 점검 |
| **input** | `data/rules/laws/*.json` (8건) |
| **output** | `data-analysis/data/processed/law_articles.parquet`<br>`data-analysis/reports/law_articles_decisions.json` |

> ⛔ **이 노트북은 조문 텍스트를 수정하지 않는다.** 손대는 순간 "출처 있는 사실"이 아니라
> "누가 적은 텍스트"가 되어 계층 1 의 존재 이유가 무너진다 (`11 §8`). **읽고 검사만 한다.**

> 📁 `data/rules/rules/*.json`(계층 2 해석 룰)은 **우리가 만든 것**이라 이 노트북 대상이 아니다.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import pandas as pd
from common import LAWS, profile, save_decisions, save_processed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 출처 | [국가법령정보 공동활용](https://open.law.go.kr) — 법제처 `lawService.do` |
| 수집 | Sprint 7 MQ-701 (`fetch_from_api`) · `KR-OSHA-ENR-126` 은 2026-08-12 추가 |
| 인증 | `LAW_API_OC` = 신청 이메일 ID 앞부분 |
| 기록 | `data/analysis/law_fetch.md` |

In [ ]:
files = sorted(LAWS.glob("*.json"))
raw = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in files}
print(f"법령 파일 {len(raw)}건")
for k in raw:
    print("  ", k)

## 2. 로드 + 기초 통계

In [ ]:
df_raw = pd.DataFrame(list(raw.values()))
print(f"원본 shape: {df_raw.shape[0]}행 × {df_raw.shape[1]}열")

In [ ]:
profile(df_raw)

In [ ]:
df_raw[["law_ref_id", "law_name", "article", "title", "fetch_status",
        "effective_from", "promulgation_no"]]

## 3. EDA

### 3-1. `fetch_status` — 무엇이 실수집됐나

In [ ]:
print(df_raw["fetch_status"].value_counts().to_string())
print()
for _, r in df_raw.iterrows():
    # ⚠ PENDING 행의 text 는 None 이 아니라 NaN(float) 으로 들어온다 — len() 이 터진다
    n = len(r["text"]) if isinstance(r["text"], str) else 0
    mark = "✅" if r["fetch_status"] == "FETCHED" else "🔶"
    print(f"{mark} {r['law_ref_id']:18} {r['law_name']} 제{r['article']}조  {n:>6,}자  {r['title']}")

### 3-2. 🚨 `조문내용` 함정 — 제목만 들어오는 조문이 있다

법제처 API 응답의 `조문내용` 은 **제목 한 줄뿐인 경우가 있다.** 그대로 쓰면 15자짜리 제목이
"조문 원문"으로 해시된다. `항·호·목` 을 **평탄화**해야 본문이 된다 — `KR-STTC-24` 가
**14자 → 3,574자**가 된 게 그 사례다(`law_fetch.md`). 아래는 그 결과가 반영돼 있는지 확인.

In [ ]:
fetched = df_raw[df_raw["fetch_status"] == "FETCHED"].copy()
fetched["글자수"] = fetched["text"].str.len()
print(fetched[["law_ref_id", "title", "글자수"]].sort_values("글자수").to_string(index=False))
print()
짧음 = fetched[fetched["글자수"] < 100]
print(f"100자 미만: {len(짧음)}건 — 평탄화 누락이 의심되면 여기부터 본다")
print(f"총 글자수 : {fetched['글자수'].sum():,}자")

### 3-3. 🚨 `text_hash` 무결성 — 텍스트가 손대졌는지 검사

⛔ **해시를 여기서 다시 구현하면 안 된다.** `text_hash` 는 단순 `sha256(text)` 가 아니라
**`"sha256:" + sha256(NFKC 정규화 + 공백 단일화)`** 다(`data/rules/engine.py`).
직접 짜면 전건 "불일치"가 나오는데 **그건 조문이 아니라 검사기가 틀린 것**이다.
→ **프로덕션과 같은 함수를 import 해서 쓴다.**

In [ ]:
sys.path.insert(0, str(LAWS.parent))   # data/rules/
from engine import normalize, text_hash   # noqa: E402  ← 프로덕션과 동일한 구현

print("normalize() 가 하는 일:", normalize.__doc__.splitlines()[0])
print("예시:", repr(normalize("  통합투자\u3000세액   공제 ")))

In [ ]:
print(f"{'law_ref_id':18}{'기록':>14}  {'재계산':>14}  판정")
print("-" * 60)
불일치 = []
for _, r in fetched.iterrows():
    calc = text_hash(r["text"])
    ok = calc == r["text_hash"]
    if not ok:
        불일치.append(r["law_ref_id"])
    print(f"{r['law_ref_id']:18}{r['text_hash'][7:17]:>14}  {calc[7:17]:>14}  "
          f"{'✅ 일치' if ok else '❌ 불일치'}")
print()
print("불일치:", 불일치 if 불일치 else "없음 — FETCHED 7건 전부 무결")

> ⚠ 해시가 어긋나면 **조문이 수집 후 편집됐다는 뜻**이다. 이 노트북은 판정만 하고
> 고치지 않는다 — 정정은 `fetch_laws.py` 재수집으로만 한다.

### 3-4. 🔶 PENDING 1건 — 왜 멈췄나

In [ ]:
pend = df_raw[df_raw["fetch_status"] != "FETCHED"]
for _, r in pend.iterrows():
    print(f"law_ref_id  : {r['law_ref_id']}")
    print(f"법령        : {r['law_name']} 제{r['article']}조")
    print(f"등록 제목   : {r['title']!r}")
    print(f"text        : {r['text']}")
    print(f"검증 메모   : {r['verification_note']}")
print()
print("→ 등록 제목이 API 값('즉시상각의 의제')과 달라 apply_fetch 가 파일을 손대기 전에 멈췄다.")
print("  D75 의 설계된 동작이다 — 자동으로 덮어쓰지 않는 것이 정상이다.")

## 4. 전처리

In [ ]:
df = df_raw.copy()
df["글자수"] = df["text"].str.len().fillna(0).astype(int)
df["수집됨"] = df["fetch_status"] == "FETCHED"
df["retrieved_at"] = pd.to_datetime(df["retrieved_at"], errors="coerce", utc=True)
df["effective_from"] = pd.to_datetime(df["effective_from"], errors="coerce")
df["해시검증"] = [
    (text_hash(t) == h) if isinstance(t, str) else None
    for t, h in zip(df["text"], df["text_hash"], strict=True)
]
# clause·effective_to·supersedes 는 전건 None — 컬럼은 남기되 사용하지 않는다
print(f"정제 후: {df.shape[0]}행 × {df.shape[1]}열")
print(f"FETCHED {df['수집됨'].sum()}건 · 해시 일치 {(df['해시검증'] == True).sum()}건")  # noqa: E712

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "law_ref_id", "used": True, "reason": "룰의 law_refs 가 참조하는 조인 키"},
    {"column": "law_name", "used": True, "reason": "인용 표기에 필요"},
    {"column": "article", "used": True, "reason": "조 번호 — 인용 앵커"},
    {"column": "title", "used": True, "reason": "조문 제목. PENDING 판정의 대조 대상이기도 하다"},
    {"column": "text", "used": True, "reason": "조문 원문 = 계층 1 사실 본체. ⛔ 편집 금지"},
    {"column": "text_hash", "used": True, "reason": "무결성 검증. 재계산과 대조해 편집 여부를 판정"},
    {"column": "fetch_status", "used": True, "reason": "FETCHED/PENDING — 인용 가능 여부를 가른다"},
    {"column": "effective_from", "used": True, "reason": "시행일. 판정 시점의 법이 맞는지 확인"},
    {"column": "promulgation_no", "used": True, "reason": "공포번호 — 버전 식별"},
    {"column": "source_url", "used": True, "reason": "출처 추적"},
    {"column": "retrieved_at", "used": True, "reason": "스냅샷 시점"},
    {"column": "verification_note", "used": True, "reason": "PENDING 사유가 여기 적힌다"},
    {"column": "clause", "used": False, "reason": "전건 None — 항 단위로 쪼갠 조문이 아직 없다"},
    {"column": "effective_to", "used": False, "reason": "전건 None — 폐지된 조문이 없다"},
    {"column": "supersedes", "used": False, "reason": "전건 None — 개정 이력 체인 미구축"},
]
assert len(column_decisions) == df_raw.shape[1]
print(f"컬럼 결정 {len(column_decisions)}건 — 원본 컬럼 수와 일치")

In [ ]:
save_decisions(
    "law_articles",
    column_decisions,
    source_url="https://open.law.go.kr",
    collected_at="2026-08-12",
    fetch_script="data/rules/fetch_laws.py",
    row_count_raw=int(df_raw.shape[0]),
    col_count_raw=int(df_raw.shape[1]),
    fetched=int(df["수집됨"].sum()),
    pending=int((~df["수집됨"]).sum()),
    total_chars=int(df["글자수"].sum()),
    hash_verified=int((df["해시검증"] == True).sum()),  # noqa: E712
    generated_columns=["글자수", "수집됨", "해시검증"],
    caveats=[
        "⛔ 조문 텍스트를 손으로 고치면 계층 1 이 무너진다 — 정정은 재수집으로만",
        "text_hash 는 'sha256:' + sha256(NFKC 정규화 + 공백 단일화) 다. 단순 sha256(text) 로 "
        "검증하면 전건 불일치가 나온다 — engine.text_hash 를 import 해서 쓸 것",
        "API 의 조문내용 은 제목뿐일 수 있다. 항·호·목 평탄화 필수 (KR-STTC-24: 14자 → 3,574자)",
        "KR-CITA-ENF-31 은 제목 불일치로 PENDING — 자동 덮어쓰기 안 하는 게 D75 설계",
        "clause·effective_to·supersedes 는 전건 None — 스키마만 있고 채워지지 않았다",
    ],
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("law_articles", df.drop(columns=["해시검증"]).assign(
    해시검증=df["해시검증"].astype("boolean")))